In [1]:
!ncu --version

NVIDIA (R) Nsight Compute Command Line Profiler
Copyright (c) 2018-2025 NVIDIA Corporation
Version 2025.1.1.0 (build 35528883) (public-release)


In [2]:
import sys, platform, subprocess, datetime
import torch

print("===== Environment Info =====")
print("OS:", platform.platform())
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA (torch built with):", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU compute capability:", torch.cuda.get_device_capability(0))
    print("GPU total memory (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

try:
    import transformers
    print("transformers:", transformers.__version__)
except ImportError:
    print("transformers: not installed")

print("\n--- nvidia-smi (driver version included")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

===== Environment Info =====
OS: Linux-6.6.122+-x86_64-with-glibc2.35
Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyTorch: 2.11.0+cu128
CUDA (torch built with): 12.8
cuDNN version: 91900
GPU available: True
GPU name: Tesla T4
GPU compute capability: (7, 5)
GPU total memory (GB): 15.637086208
transformers: 5.16.1

--- nvidia-smi (driver version included
Wed Sep  9 11:11:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+===

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/KV Cache/4_Nsight_analysis/"

In [ ]:
!ncu \
    --nvtx \
    --nvtx-include "QK_MATMUL/" \
    --replay-mode application \
    --section SpeedOfLight \
    -o KV_cache_case_SpeedOfLight \
    python KV_cache_Nsight_analysis_test_file.py

==PROF== Connected to process 1131 (/usr/bin/python3.12)
loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
config.json: 100% 665/665 [00:00<00:00, 3.93MB/s]

model.safetensors: downloading bytes:  46% 253M/548M [00:01<00:01, 220MB/s, 21.1MB/s  ]
model.safetensors: downloading bytes:  71% 390M/548M [00:02<00:00, 173MB/s, 33.4MB/s  ]
model.safetensors: downloading bytes:  87% 474M/548M [00:03<00:00, 151MB/s, 40.6MB/s  ]
model.safetensors: reconstructing file:  73% 402M/548M [00:03<00:00, 155MB/s, 24.9MB/s  ]
model.safetensors: downloading bytes: 100% 474M/474M [00:04<00:00, 103MB/s, 40.4MB/s  ]
model.safetensors: reconstructing file: 100% 548M/548M [00:04<00:00, 119MB/s, 44.9MB/s  ]
Loading weights: 100% 148/148 [00:00<00:00, 10503.68it/s]
generation_config.json: 100% 124/124 [00:00<00:00, 559kB/s]
===== running_mode = inf_cache =====
==PROF== Profiling "elementwise_kernel" - 0: Appl

In [ ]:
!ncu \
    --nvtx \
    --nvtx-include "QK_MATMUL/" \
    --replay-mode application \
    --section MemoryWorkloadAnalysis \
    --section WarpStateStats \
    -o KV_cache_Memory_Warp \
    python KV_cache_Nsight_analysis_test_file.py

==PROF== Connected to process 2497 (/usr/bin/python3.12)
loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
Loading weights: 100% 148/148 [00:00<00:00, 5244.70it/s]
===== running_mode = inf_cache =====
==PROF== Profiling "elementwise_kernel" - 0: Application replay pass 1
==PROF== Profiling "elementwise_kernel" - 1: Application replay pass 1
==PROF== Profiling "elementwise_kernel" - 2: Application replay pass 1
==PROF== Profiling "kernel" - 3: Application replay pass 1
1014 Token process...
[inf_cache] step 1014: next tokens = ['\n', 'ide', ' the', ',', ',', ' I', ' your', '?', ' much', ',', '\n', '\n', ' me', ' the', 'US', "'"]
1015 Token process...
[inf_cache] step 1015: next tokens = ['p', ',', ' right', ' and', ' and', ' will', ' cause', '\n', ' as', ' I', '\n', '\n', ',', ' way', ':', 'er']
1016 Token process...
[inf_cache] step 1016: next tokens = ['raise', ' and', ',', ' the'

In [ ]:
!ncu --import KV_cache_case_SpeedOfLight.ncu-rep --page details

[1131] python3.12@127.0.0.1
  void at::elementwise_kernel<128, 4, void at::gpu_kernel_impl_nocast<at::bfloat16_copy_kernel_cuda(at::TensorIteratorBase &)::[lambda(float) (instance 1)]>(at::TensorIteratorBase &, const T1 &)::[lambda(int) (instance 1)]>(int, T3) (24552, 1, 1)x(128, 1, 1), Context 1, Stream 7, Device 0, CC 7.5

    NVTX Push/Pop Stack for Thread 1131:
      <default domain>
        <0,QK_MATMUL>
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.99
    SM Frequency                    Mhz       584.98
    Elapsed Cycles                cycle      222,541
    Memory Throughput                 %        74.53
    DRAM Throughput                   %        74.53
    Duration                         us       380.42
    L1/TEX Cache Throughput           %        24.48
    L2 Cache T

In [ ]:
!ncu --import KV_cache_Memory_Warp.ncu-rep --page details

[2497] python3.12@127.0.0.1
  void at::elementwise_kernel<128, 4, void at::gpu_kernel_impl_nocast<at::bfloat16_copy_kernel_cuda(at::TensorIteratorBase &)::[lambda(float) (instance 1)]>(at::TensorIteratorBase &, const T1 &)::[lambda(int) (instance 1)]>(int, T3) (24552, 1, 1)x(128, 1, 1), Context 1, Stream 7, Device 0, CC 7.5

    NVTX Push/Pop Stack for Thread 2497:
      <default domain>
        <0,QK_MATMUL>
    Section: Memory Workload Analysis
    ----------------- ----------- ------------
    Metric Name       Metric Unit Metric Value
    ----------------- ----------- ------------
    Memory Throughput     Gbyte/s       237.92
    Mem Busy                    %        22.70
    Max Bandwidth               %        74.39
    L1/TEX Hit Rate             %        16.65
    L2 Hit Rate                 %        33.47
    Mem Pipes Busy              %        19.83
    ----------------- ----------- ------------

    Section: Warp State Statistics
    ---------------------------------------

In [ ]:
!ncu \
    --nvtx \
    --nvtx-include "QK_MATMUL/" \
    --replay-mode application \
    --set full \
    -o KV_cache_Arithmentic_intensity \
    python KV_cache_Nsight_analysis_test_file.py

==PROF== Connected to process 4497 (/usr/bin/python3.12)
loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
Loading weights: 100% 148/148 [00:00<00:00, 6491.17it/s]
===== running_mode = inf_cache =====
==PROF== Profiling "elementwise_kernel" - 0: Application replay pass 1
==PROF== Profiling "elementwise_kernel" - 1: Application replay pass 1
==PROF== Profiling "elementwise_kernel" - 2: Application replay pass 1
==PROF== Profiling "kernel" - 3: Application replay pass 1
1014 Token process...
[inf_cache] step 1014: next tokens = ['\n', 'ide', ' the', ',', ',', ' I', ' your', '?', ' much', ',', '\n', '\n', ' me', ' the', 'US', "'"]
1015 Token process...
[inf_cache] step 1015: next tokens = ['p', ',', ' right', ' and', ' and', ' will', ' cause', '\n', ' as', ' I', '\n', '\n', ',', ' way', ':', 'er']
1016 Token process...
[inf_cache] step 1016: next tokens = ['raise', ' and', ',', ' the'

In [ ]:
!ncu \
    --nvtx \
    --nvtx-include "QK_MATMUL/" \
    --replay-mode application \
    --section SpeedOfLight \
    -o No_KV_cache_case_SpeedOfLight \
    python No_KV_cache_Nsight_analysis_test_file.py

==PROF== Connected to process 1657 (/usr/bin/python3.12)
loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
config.json: 100% 665/665 [00:00<00:00, 2.71MB/s]

model.safetensors: downloading bytes:  58% 316M/548M [00:02<00:01, 161MB/s, 27.2MB/s  ]
model.safetensors: downloading bytes:  87% 474M/548M [00:02<00:00, 224MB/s, 40.6MB/s  ]
model.safetensors: reconstructing file:  73% 402M/548M [00:03<00:01, 139MB/s, 25.3MB/s  ]
model.safetensors: downloading bytes: 100% 474M/474M [00:03<00:00, 147MB/s, 41.1MB/s  ]
model.safetensors: reconstructing file: 100% 548M/548M [00:03<00:00, 170MB/s, 49.3MB/s  ]
Loading weights: 100% 148/148 [00:00<00:00, 4595.07it/s]
generation_config.json: 100% 124/124 [00:00<00:00, 493kB/s]
===== running_mode = inf_no_cache =====
1014 Token process...
[inf_no_cache] step 1014: next tokens = ['\n', 'ide', ' the', ',', ',', ' I', ' your', '?', ' much', ',', '\n', '

In [6]:
!ncu --import No_KV_cache_case_SpeedOfLight.ncu-rep --page details

[1657] python3.12@127.0.0.1
  void at::elementwise_kernel<128, 4, void at::gpu_kernel_impl_nocast<at::direct_copy_kernel_cuda(at::TensorIteratorBase &)::[lambda() (instance 3)]::operator ()() lambda() (instance 12)]::operator ()() lambda(c10::BFloat16) (instance 1)]>(at::TensorIteratorBase &, const T1 &)::[lambda(int) (instance 1)]>(int, T3) (24552, 1, 1)x(128, 1, 1), Context 1, Stream 7, Device 0, CC 7.5

    NVTX Push/Pop Stack for Thread 1657:
      <default domain>
        <0,QK_MATMUL>
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         5.00
    SM Frequency                    Mhz       584.97
    Elapsed Cycles                cycle      247,061
    Memory Throughput                 %        53.99
    DRAM Throughput                   %        53.99
    Duration                         